# xgboost_walk_forward_validation_tuning

XGBoost validation tuning using the full-history session-aligned dataset.

The notebook mirrors the other model-validation pipelines: it builds compact feature-transformation and XGBoost hyperparameter grids, splits on the full session calendar while keeping neutral targets as the middle class, selects configurations with expanding-window walk-forward validation, scores the selected multiclass configuration on the holdout validation period, and evaluates the frozen model on the untouched test split.

Requires the `xgboost` Python package. If it is missing, install it in the notebook environment before running the model-fitting cells.


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    XGBClassifier = None
    XGBOOST_IMPORT_ERROR = exc
else:
    XGBOOST_IMPORT_ERROR = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)


In [2]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_utils.experiment_config import build_default_config
from notebook_utils.feature_set_grid_builder import FeatureFrameBuilder, FeatureSetGridBuilder
from notebook_utils.metrics import ClassificationMetrics
from notebook_utils.model_report_builder import ModelReportBuilder
from notebook_utils.split_utils import make_split_dates, make_walk_forward_fold_specs, subset_by_dates

CONFIG = build_default_config(PROJECT_ROOT)

XGB_PARAM_GRID = [
    {
        "param_set": "xgb_150_depth2_lr0p03_sub0p8_col0p8_child10_l2_5_balanced",
        "n_estimators": 150,
        "max_depth": 2,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 10,
        "reg_lambda": 5.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "xgb_250_depth2_lr0p02_sub0p8_col0p8_child10_l2_10_balanced",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.02,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 10,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "xgb_150_depth3_lr0p03_sub0p8_col0p8_child15_l2_5_balanced",
        "n_estimators": 150,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 15,
        "reg_lambda": 5.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "xgb_250_depth3_lr0p02_sub0p8_col0p8_child15_l2_10_balanced",
        "n_estimators": 250,
        "max_depth": 3,
        "learning_rate": 0.02,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 15,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "xgb_400_depth2_lr0p01_sub0p7_col0p8_child20_l1_0p5_l2_10_balanced",
        "n_estimators": 400,
        "max_depth": 2,
        "learning_rate": 0.01,
        "subsample": 0.7,
        "colsample_bytree": 0.8,
        "min_child_weight": 20,
        "reg_lambda": 10.0,
        "reg_alpha": 0.5,
        "gamma": 0.0,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "xgb_400_depth3_lr0p01_sub0p7_col0p7_child20_l1_0p5_l2_15_balanced",
        "n_estimators": 400,
        "max_depth": 3,
        "learning_rate": 0.01,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "min_child_weight": 20,
        "reg_lambda": 15.0,
        "reg_alpha": 0.5,
        "gamma": 0.0,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "xgb_150_depth4_lr0p03_sub0p8_col0p7_child20_gamma1_l2_10_balanced",
        "n_estimators": 150,
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.7,
        "min_child_weight": 20,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0,
        "gamma": 1.0,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "xgb_250_depth4_lr0p02_sub0p7_col0p7_child30_gamma1_l2_15_balanced",
        "n_estimators": 250,
        "max_depth": 4,
        "learning_rate": 0.02,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "min_child_weight": 30,
        "reg_lambda": 15.0,
        "reg_alpha": 0.0,
        "gamma": 1.0,
        "class_weight_mode": "balanced",
    },
    {
        "param_set": "xgb_250_depth2_lr0p02_sub0p8_col0p8_child10_l2_10_unweighted",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.02,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 10,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "class_weight_mode": "none",
    },
    {
        "param_set": "xgb_250_depth3_lr0p02_sub0p8_col0p8_child15_l2_10_unweighted",
        "n_estimators": 250,
        "max_depth": 3,
        "learning_rate": 0.02,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 15,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
        "class_weight_mode": "none",
    },
]

pd.DataFrame(XGB_PARAM_GRID)

feature_grid = FeatureSetGridBuilder.build(
    max_features_per_model=CONFIG["max_features_per_model"],
)

PRICE_FEATURES = feature_grid.price_features
VOLUME_FEATURE_OPTIONS = feature_grid.volume_feature_options
GDELT_FEATURE_OPTIONS = feature_grid.gdelt_feature_options
GDELT_SENTIMENT_FEATURE_OPTIONS = feature_grid.gdelt_sentiment_feature_options
GDELT_ATTENTION_FEATURE_OPTIONS = feature_grid.gdelt_attention_feature_options
REDDIT_FEATURE_OPTIONS = feature_grid.reddit_feature_options
REDDIT_ATTENTION_FEATURE_OPTIONS = feature_grid.reddit_attention_feature_options
GOOGLE_TRENDS_FEATURE_OPTIONS = feature_grid.google_trends_feature_options
GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS = feature_grid.google_score_attention_feature_options
DERIVED_FEATURE_COLUMNS = feature_grid.derived_feature_columns
BASE_VOLUME_OPTION = feature_grid.base_volume_option
BASELINE_FEATURE_SET = feature_grid.baseline_feature_set
FEATURE_SET_SPECS = feature_grid.feature_set_specs
FEATURE_SETS = feature_grid.feature_sets
FEATURE_SET_METADATA = feature_grid.feature_set_metadata
SKIPPED_FEATURE_SETS = feature_grid.skipped_feature_sets
FEATURE_SETS_TO_TEST = feature_grid.feature_sets_to_test
pd.DataFrame(XGB_PARAM_GRID)


,param_set,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,reg_lambda,reg_alpha,gamma,class_weight_mode
0,xgb_150_depth2_lr0p03_sub0p8_col0p8_child10_l2...,150,2,0.03,0.8,0.8,10,5.0,0.0,0.0,balanced
1,xgb_250_depth2_lr0p02_sub0p8_col0p8_child10_l2...,250,2,0.02,0.8,0.8,10,10.0,0.0,0.0,balanced
2,xgb_150_depth3_lr0p03_sub0p8_col0p8_child15_l2...,150,3,0.03,0.8,0.8,15,5.0,0.0,0.0,balanced
3,xgb_250_depth3_lr0p02_sub0p8_col0p8_child15_l2...,250,3,0.02,0.8,0.8,15,10.0,0.0,0.0,balanced
4,xgb_400_depth2_lr0p01_sub0p7_col0p8_child20_l1...,400,2,0.01,0.7,0.8,20,10.0,0.5,0.0,balanced
5,xgb_400_depth3_lr0p01_sub0p7_col0p7_child20_l1...,400,3,0.01,0.7,0.7,20,15.0,0.5,0.0,balanced
6,xgb_150_depth4_lr0p03_sub0p8_col0p7_child20_ga...,150,4,0.03,0.8,0.7,20,10.0,0.0,1.0,balanced
7,xgb_250_depth4_lr0p02_sub0p7_col0p7_child30_ga...,250,4,0.02,0.7,0.7,30,15.0,0.0,1.0,balanced
8,xgb_250_depth2_lr0p02_sub0p8_col0p8_child10_l2...,250,2,0.02,0.8,0.8,10,10.0,0.0,0.0,none
9,xgb_250_depth3_lr0p02_sub0p8_col0p8_child15_l2...,250,3,0.02,0.8,0.8,15,10.0,0.0,0.0,none


In [3]:
from __future__ import annotations


def sample_weight_from_target(y_train: pd.Series, mode: str | None) -> np.ndarray | None:
    if mode in [None, "none"]:
        return None
    if mode != "balanced":
        raise ValueError(f"Unsupported class_weight_mode for XGBoost: {mode}")
    counts = pd.Series(y_train).value_counts()
    n_total = float(len(y_train))
    n_classes = float(len(counts))
    if n_classes <= 0.0:
        return None
    weights_by_class = {
        int(class_value): n_total / (n_classes * float(class_count))
        for class_value, class_count in counts.items()
        if class_count > 0
    }
    return pd.Series(y_train).map(weights_by_class).to_numpy(dtype=float)


def build_xgb_pipeline_from_params(params: dict) -> Pipeline:
    if XGBClassifier is None:
        raise ImportError(
            "xgboost is not installed in this Python environment. Install it with `py -3.8 -m pip install xgboost` before running the XGBoost notebook."
        ) from XGBOOST_IMPORT_ERROR

    xgb_params = dict(params)
    xgb_params.pop("param_set", None)
    xgb_params.pop("class_weight_mode", None)
    xgb_params.setdefault("objective", "multi:softprob")
    xgb_params.setdefault("num_class", len(ClassificationMetrics.CLASS_VALUES))
    xgb_params.setdefault("eval_metric", "mlogloss")
    xgb_params.setdefault("tree_method", "hist")
    xgb_params.setdefault("random_state", CONFIG["random_state"])
    xgb_params.setdefault("n_jobs", -1)
    xgb_params.setdefault("verbosity", 0)
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(**xgb_params)),
        ]
    )

In [4]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = FeatureFrameBuilder.build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)
walk_forward_fold_specs = make_walk_forward_fold_specs(
    train_dates,
    n_folds=CONFIG["walk_forward_folds"],
    validation_size=CONFIG["walk_forward_validation_dates"],
    min_train_dates=CONFIG["walk_forward_min_train_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {
    "train": train_dates,
    "validation": validation_dates,
    "test": test_dates,
}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin(ClassificationMetrics.CLASS_VALUES)].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)
train_validation_df = subset_by_dates(modeled_df, list(train_dates) + list(validation_dates))

split_summary_rows = []
for split_name in ["train", "validation", "test"]:
    all_split_df = feature_df[feature_df["split"].eq(split_name)]
    modeled_split_df = modeled_df[modeled_df["split"].eq(split_name)]
    class_rates = modeled_split_df["target"].value_counts(normalize=True)
    split_summary_rows.append(
        {
            "split": split_name,
            "session_rows": len(all_split_df),
            "modeled_rows": len(modeled_split_df),
            "session_dates": all_split_df["date"].nunique(),
            "modeled_dates": modeled_split_df["date"].nunique(),
            "date_min": all_split_df["date"].min(),
            "date_max": all_split_df["date"].max(),
            "target_down_rate": float(class_rates.get(0, 0.0)),
            "target_neutral_rate": float(class_rates.get(1, 0.0)),
            "target_up_rate": float(class_rates.get(2, 0.0)),
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
walk_forward_fold_summary_df = pd.DataFrame(
    [
        {
            "fold": spec["fold"],
            "train_n_dates": spec["train_n_dates"],
            "train_date_min": spec["train_date_min"],
            "train_date_max": spec["train_date_max"],
            "validation_n_dates": spec["validation_n_dates"],
            "validation_date_min": spec["validation_date_min"],
            "validation_date_max": spec["validation_date_max"],
        }
        for spec in walk_forward_fold_specs
    ]
)

split_summary_df

,split,session_rows,modeled_rows,session_dates,modeled_dates,date_min,date_max,target_down_rate,target_neutral_rate,target_up_rate
0,train,5632,5632,704,704,2021-01-04,2023-10-19,0.386541,0.208629,0.404830
1,validation,1880,1880,235,235,2023-10-23,2024-09-27,0.327128,0.242553,0.430319
2,test,2512,2504,314,313,2024-10-01,2025-12-31,0.356629,0.246805,0.396565


In [5]:
walk_forward_fold_summary_df

,fold,train_n_dates,train_date_min,train_date_max,validation_n_dates,validation_date_min,validation_date_max
0,1,383,2021-01-04,2022-07-12,80,2022-07-14,2022-11-03
1,2,463,2021-01-04,2022-11-02,80,2022-11-04,2023-03-02
2,3,543,2021-01-04,2023-03-01,80,2023-03-03,2023-06-27
3,4,623,2021-01-04,2023-06-26,80,2023-06-28,2023-10-19


In [6]:
neutral_summary_by_ticker_df = (
    feature_df.groupby("ticker")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reset_index()
)
neutral_summary_by_ticker_df["neutral_rate_among_available"] = (
    neutral_summary_by_ticker_df["neutral"] / neutral_summary_by_ticker_df["target_available"]
)
neutral_summary_by_ticker_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_ticker_df["neutral_rate_among_available"]

neutral_summary_by_split_df = (
    feature_df[feature_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
neutral_summary_by_split_df["neutral_rate_among_available"] = (
    neutral_summary_by_split_df["neutral"] / neutral_summary_by_split_df["target_available"]
)
neutral_summary_by_split_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_split_df["neutral_rate_among_available"]

print("Neutral coverage by split")
print(neutral_summary_by_split_df.to_string(index=False))
print("\nNeutral coverage by ticker")
neutral_summary_by_ticker_df

Neutral coverage by split
     split  rows  target_available  neutral  neutral_rate_among_available  modeled_rate_among_available
     train  5632              5632     1175                      0.208629                      0.791371
validation  1880              1880      456                      0.242553                      0.757447
      test  2512              2504      618                      0.246805                      0.753195

Neutral coverage by ticker


,ticker,rows,target_available,neutral,neutral_rate_among_available,modeled_rate_among_available
0,AAPL,1255,1254,378,0.301435,0.698565
1,AMD,1255,1254,220,0.175439,0.824561
2,AMZN,1255,1254,300,0.239234,0.760766
3,GOOGL,1255,1254,319,0.254386,0.745614
4,META,1255,1254,277,0.220893,0.779107
5,MSFT,1255,1254,400,0.318979,0.681021
6,NVDA,1255,1254,173,0.137959,0.862041
7,TSLA,1255,1254,184,0.146730,0.853270


In [7]:
missing_requested_feature_sets = [name for name in FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f"Unknown feature sets: {missing_requested_feature_sets}")

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(features),
            "features": features,
        }
        for feature_set_name, features in FEATURE_SETS.items()
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

skipped_feature_sets_df = pd.DataFrame(SKIPPED_FEATURE_SETS)

print(f"Selection metric: {CONFIG['selection_metric']}")
print(f"Primary validation metric: {CONFIG['primary_validation_metric']}")
print(f"Walk-forward folds: {len(walk_forward_fold_specs)}")
print(f"Target classes: {ClassificationMetrics.CLASS_LABELS}")
print("Prediction rule: multiclass argmax")
print(f"Max features per model: {CONFIG['max_features_per_model']}")
print(f"Feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets: {sum('attention' in FEATURE_SET_METADATA[name]['feature_family'] for name in FEATURE_SETS_TO_TEST)}")
print(f"Skipped feature sets above max feature limit: {len(SKIPPED_FEATURE_SETS)}")
print(f"XGBoost parameter sets: {len(XGB_PARAM_GRID)}")
print(f"Walk-forward validation fits: {len(FEATURE_SETS_TO_TEST) * len(XGB_PARAM_GRID) * len(walk_forward_fold_specs)}")

candidate_feature_sets_df


Selection metric: balanced_accuracy
Primary validation metric: balanced_accuracy
Walk-forward folds: 4
Target classes: {0: 'down', 1: 'neutral', 2: 'up'}
Prediction rule: multiclass argmax
Max features per model: 14
Feature sets to test: 80
Attention feature sets: 38
Skipped feature sets above max feature limit: 0
XGBoost parameter sets: 10
Walk-forward validation fits: 3200


,feature_set,feature_family,n_features,features
0,Model B - price + volume | volume log1p zscore...,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
1,Model B - price + volume | volume percentile r...,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
2,Model B - price + volume | volume zscore 10d,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
3,Model B - price + volume | volume zscore 20d,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
4,Model B - price + volume | volume zscore 20d c...,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
...,...,...,...,...
75,Model F - price + volume + all alternative dat...,price + volume + all alternative data,14,"[return_1d, return_5d, return_20d, rolling_vol..."
76,Model N - price + volume + all attention | per...,price + volume + all attention,12,"[return_1d, return_5d, return_20d, rolling_vol..."
77,Model N - price + volume + all attention | zsc...,price + volume + all attention,12,"[return_1d, return_5d, return_20d, rolling_vol..."
78,Model N - price + volume + all attention | zsc...,price + volume + all attention,12,"[return_1d, return_5d, return_20d, rolling_vol..."


In [8]:
from __future__ import annotations


XGB_PARAM_COLUMNS = [
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "reg_lambda",
    "reg_alpha",
    "gamma",
    "class_weight_mode",
]


def prepare_train_eval_feature_frames(
    features: list[str],
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "google_trends_above_ticker_train_median" in features:
        return FeatureFrameBuilder.add_google_trends_train_median_feature(train_input_df, eval_input_df)
    return train_input_df, eval_input_df


def params_from_result_row(row: dict | pd.Series) -> dict:
    return {
        "param_set": row["param_set"],
        "n_estimators": int(row["n_estimators"]),
        "max_depth": int(row["max_depth"]),
        "learning_rate": float(row["learning_rate"]),
        "subsample": float(row["subsample"]),
        "colsample_bytree": float(row["colsample_bytree"]),
        "min_child_weight": int(row["min_child_weight"]),
        "reg_lambda": float(row["reg_lambda"]),
        "reg_alpha": float(row["reg_alpha"]),
        "gamma": float(row["gamma"]),
        "class_weight_mode": row.get("class_weight_mode", "balanced"),
    }

def add_param_columns(row: dict, params: dict, param_columns: list[str]) -> None:
    for column in param_columns:
        row[column] = params.get(column)


def evaluate_xgb_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    train_features_df, eval_features_df = prepare_train_eval_feature_frames(
        features,
        train_input_df,
        eval_input_df,
    )

    sample_weight = sample_weight_from_target(train_features_df["target"], params.get("class_weight_mode", "balanced"))
    fit_kwargs = {"model__sample_weight": sample_weight} if sample_weight is not None else {}
    pipeline = build_xgb_pipeline_from_params(params)
    pipeline.fit(train_features_df[features], train_features_df["target"], **fit_kwargs)
    probabilities = pipeline.predict_proba(eval_features_df[features])
    classes = pipeline.named_steps["model"].classes_
    preds = ClassificationMetrics.predictions_from_probabilities(probabilities, classes)

    metric_result = ClassificationMetrics.metrics_from_predictions(eval_features_df["target"], preds)
    preds = metric_result.pop("preds")
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    row = {
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        **metric_result,
    }
    add_param_columns(row, params, XGB_PARAM_COLUMNS)

    if not return_predictions:
        return row

    predictions_df = eval_features_df[["date", "ticker", "target"]].copy()
    for column, values in ClassificationMetrics.probability_column_dict(probabilities, classes).items():
        predictions_df[column] = values
    predictions_df["prediction"] = preds
    return row, predictions_df


def evaluate_xgb_config_walk_forward(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    modeled_input_df: pd.DataFrame,
    fold_specs: list[dict],
) -> tuple[dict, list[dict]]:
    fold_rows = []
    for spec in fold_specs:
        fold_row = evaluate_xgb_params(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            train_input_df=subset_by_dates(modeled_input_df, spec["train_dates"]),
            eval_input_df=subset_by_dates(modeled_input_df, spec["validation_dates"]),
            split_name="walk_forward_validation",
        )
        fold_rows.append(
            {
                **fold_row,
                "fold": spec["fold"],
                "fold_train_n_dates": spec["train_n_dates"],
                "fold_validation_n_dates": spec["validation_n_dates"],
                "fold_train_date_min": spec["train_date_min"],
                "fold_train_date_max": spec["train_date_max"],
                "fold_validation_date_min": spec["validation_date_min"],
                "fold_validation_date_max": spec["validation_date_max"],
            }
        )

    fold_results_df = pd.DataFrame(fold_rows)
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    metric_means = {
        metric: float(fold_results_df[metric].mean())
        for metric in ['accuracy', 'balanced_accuracy', 'f1_score', 'f1_weighted']
    }
    summary_row = {
        "split": "walk_forward_validation",
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        **metric_means,
    }
    add_param_columns(summary_row, params, XGB_PARAM_COLUMNS)
    return summary_row, fold_rows

In [9]:
selection_metric = CONFIG["selection_metric"]
walk_forward_grid_rows = []
walk_forward_fold_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in XGB_PARAM_GRID:
        summary_row, fold_rows = evaluate_xgb_config_walk_forward(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            modeled_input_df=modeled_df,
            fold_specs=walk_forward_fold_specs,
        )
        walk_forward_grid_rows.append(summary_row)
        walk_forward_fold_rows.extend(fold_rows)

walk_forward_grid_results_df = pd.DataFrame(walk_forward_grid_rows)
walk_forward_fold_results_df = pd.DataFrame(walk_forward_fold_rows)
if selection_metric not in walk_forward_grid_results_df.columns:
    raise KeyError(f"Selection metric is not available: {selection_metric}")

validation_grid_results_df = walk_forward_grid_results_df.sort_values(
    [selection_metric, "balanced_accuracy", "f1_score", "accuracy", "feature_set", "param_set"],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)


In [10]:
validation_best_by_feature_set_df = ModelReportBuilder.select_best_validation_by_feature_set(
    validation_grid_results_df,
    selection_metric=CONFIG["selection_metric"],
)

validation_best_by_feature_set_report_df = ModelReportBuilder.build_validation_best_by_feature_set_report(
    validation_best_by_feature_set_df,
    param_columns=XGB_PARAM_COLUMNS,
)

validation_best_by_feature_set_report_df


,feature_family,feature_set,n_features,param_set,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,reg_lambda,reg_alpha,gamma,class_weight_mode,validation_accuracy,validation_balanced_accuracy,validation_f1_score,validation_f1_weighted
0,price + volume + Reddit,Model E - price + volume + Reddit | Reddit sen...,12,xgb_250_depth2_lr0p02_sub0p8_col0p8_child10_l2...,250,2,0.02,0.8,0.8,10,10.0,0.0,0.0,balanced,0.360938,0.369780,0.338696,0.347687
1,price + volume + Reddit sentiment lags,Model R - price + volume + Reddit sentiment la...,12,xgb_250_depth2_lr0p02_sub0p8_col0p8_child10_l2...,250,2,0.02,0.8,0.8,10,10.0,0.0,0.0,balanced,0.360938,0.369780,0.338696,0.347687
2,price + volume + Reddit,Model E - price + volume + Reddit | Reddit sen...,11,xgb_250_depth4_lr0p02_sub0p7_col0p7_child30_ga...,250,4,0.02,0.7,0.7,30,15.0,0.0,1.0,balanced,0.359375,0.367528,0.339756,0.346384
3,price + volume + Reddit,Model E - price + volume + Reddit | Reddit zsc...,11,xgb_150_depth4_lr0p03_sub0p8_col0p7_child20_ga...,150,4,0.03,0.8,0.7,20,10.0,0.0,1.0,balanced,0.360547,0.367331,0.341036,0.348576
4,price + volume + GDELT,Model C - price + volume + GDELT | GDELT perce...,11,xgb_150_depth4_lr0p03_sub0p8_col0p7_child20_ga...,150,4,0.03,0.8,0.7,20,10.0,0.0,1.0,balanced,0.358984,0.366280,0.339972,0.349188
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,price + volume + GDELT + Reddit,Model D - price + volume + GDELT + Reddit | la...,13,xgb_250_depth3_lr0p02_sub0p8_col0p8_child15_l2...,250,3,0.02,0.8,0.8,15,10.0,0.0,0.0,balanced,0.346094,0.355015,0.324763,0.332196
76,price + volume + Reddit attention,Model K - price + volume + Reddit attention | ...,11,xgb_150_depth4_lr0p03_sub0p8_col0p7_child20_ga...,150,4,0.03,0.8,0.7,20,10.0,0.0,1.0,balanced,0.349609,0.354829,0.328684,0.338986
77,price + volume + GDELT attention,Model J - price + volume + GDELT attention | G...,11,xgb_150_depth4_lr0p03_sub0p8_col0p7_child20_ga...,150,4,0.03,0.8,0.7,20,10.0,0.0,1.0,balanced,0.348828,0.354564,0.328362,0.337277
78,price + volume + Google,Model G - price + volume + Google | Google zsc...,10,xgb_250_depth4_lr0p02_sub0p7_col0p7_child30_ga...,250,4,0.02,0.7,0.7,30,15.0,0.0,1.0,balanced,0.346094,0.354525,0.325754,0.332591


In [11]:
best_validation_params_df = validation_best_by_feature_set_df.copy()


In [12]:
validation_refit_rows = []
test_rows = []

for row in best_validation_params_df.to_dict(orient="records"):
    params = params_from_result_row(row)
    feature_set_name = row["feature_set"]
    validation_refit_row = evaluate_xgb_params(
        feature_set_name=feature_set_name,
        features=FEATURE_SETS[feature_set_name],
        params=params,
        train_input_df=train_df,
        eval_input_df=validation_df,
        split_name="validation_refit_train",
    )
    validation_refit_rows.append(validation_refit_row)
    test_rows.append(
        evaluate_xgb_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_validation_df,
            eval_input_df=test_df,
            split_name="test_refit_train_validation",
        )
    )

validation_refit_results_df = pd.DataFrame(validation_refit_rows)
test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ["balanced_accuracy", "f1_score", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

(
    simple_hyperparameter_summary_df,
    baseline_walk_forward_row,
    baseline_validation_refit_row,
    baseline_test_row,
) = ModelReportBuilder.build_simple_hyperparameter_summary(
    best_validation_params_df=best_validation_params_df,
    validation_refit_results_df=validation_refit_results_df,
    test_best_validation_params_df=test_best_validation_params_df,
    baseline_feature_set=BASELINE_FEATURE_SET,
)

In [13]:
validation_selected_family_test_report_df = ModelReportBuilder.build_validation_selected_family_test_report(
    simple_hyperparameter_summary_df,
    param_columns=XGB_PARAM_COLUMNS,
)

final_test_verification_path = ModelReportBuilder.save_final_test_verification(
    validation_selected_family_test_report_df,
    model_name="xgboost",
    output_dir=PROJECT_ROOT / "notebooks" / "outputs-three-classes-rich-price",
)
print(f"Saved final test verification to: {final_test_verification_path}")

validation_selected_family_test_report_df


Saved final test verification to: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks\outputs-three-classes-rich-price\xgboost_final_test_verification.csv


,feature_family,feature_set,n_features,param_set,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,reg_lambda,reg_alpha,gamma,class_weight_mode,validation_accuracy,validation_balanced_accuracy,validation_f1_score,validation_f1_weighted,test_accuracy,test_balanced_accuracy,test_f1_score,test_f1_weighted,price_volume_baseline_feature_set,price_volume_baseline_test_balanced_accuracy,test_balanced_accuracy_change_vs_price_volume
0,price + volume + GDELT + Reddit,Model D - price + volume + GDELT + Reddit | zs...,13,xgb_250_depth4_lr0p02_sub0p7_col0p7_child30_ga...,250,4,0.02,0.7,0.7,30,15.0,0.0,1.0,balanced,0.360156,0.363198,0.338552,0.348883,0.387380,0.413029,0.386127,0.381463,Model B - price + volume | volume percentile r...,0.395999,0.017031
1,price + volume + Google,Model G - price + volume + Google | Google zsc...,10,xgb_250_depth4_lr0p02_sub0p7_col0p7_child30_ga...,250,4,0.02,0.7,0.7,30,15.0,0.0,1.0,balanced,0.354297,0.363485,0.334699,0.339922,0.385383,0.411479,0.384017,0.378186,Model B - price + volume | volume percentile r...,0.395999,0.015481
2,price + volume + Google score attention,Model L - price + volume + Google score attent...,10,xgb_250_depth4_lr0p02_sub0p7_col0p7_child30_ga...,250,4,0.02,0.7,0.7,30,15.0,0.0,1.0,balanced,0.354297,0.363485,0.334699,0.339922,0.385383,0.411479,0.384017,0.378186,Model B - price + volume | volume percentile r...,0.395999,0.015481
3,price + volume + GDELT sentiment lags,Model Q - price + volume + GDELT sentiment lag...,12,xgb_250_depth4_lr0p02_sub0p7_col0p7_child30_ga...,250,4,0.02,0.7,0.7,30,15.0,0.0,1.0,balanced,0.355469,0.364563,0.334646,0.340949,0.378994,0.407067,0.376855,0.370730,Model B - price + volume | volume percentile r...,0.395999,0.011069
4,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,12,xgb_150_depth3_lr0p03_sub0p8_col0p8_child15_l2...,150,3,0.03,0.8,0.8,15,5.0,0.0,0.0,balanced,0.350781,0.361222,0.330845,0.337113,0.374601,0.404650,0.371804,0.363706,Model B - price + volume | volume percentile r...,0.395999,0.008652
5,price + volume + Reddit attention,Model K - price + volume + Reddit attention | ...,10,xgb_250_depth4_lr0p02_sub0p7_col0p7_child30_ga...,250,4,0.02,0.7,0.7,30,15.0,0.0,1.0,balanced,0.355078,0.362416,0.334338,0.341638,0.375799,0.402950,0.374149,0.367991,Model B - price + volume | volume percentile r...,0.395999,0.006951
6,price + volume + all alternative data,Model F - price + volume + all alternative dat...,14,xgb_150_depth4_lr0p03_sub0p8_col0p7_child20_ga...,150,4,0.03,0.8,0.7,20,10.0,0.0,1.0,balanced,0.359375,0.363827,0.338009,0.348834,0.375799,0.402452,0.374426,0.367858,Model B - price + volume | volume percentile r...,0.395999,0.006453
7,price + volume + Reddit + Google,Model I - price + volume + Reddit + Google | z...,12,xgb_250_depth3_lr0p02_sub0p8_col0p8_child15_l2...,250,3,0.02,0.8,0.8,15,10.0,0.0,0.0,balanced,0.350000,0.362141,0.328824,0.333280,0.371006,0.400134,0.368059,0.360168,Model B - price + volume | volume percentile r...,0.395999,0.004136
8,price + volume + GDELT sentiment + Reddit atte...,Model O - price + volume + GDELT sentiment + R...,11,xgb_150_depth4_lr0p03_sub0p8_col0p7_child20_ga...,150,4,0.03,0.8,0.7,20,10.0,0.0,1.0,balanced,0.359766,0.366210,0.340057,0.348632,0.371805,0.399186,0.370024,0.363715,Model B - price + volume | volume percentile r...,0.395999,0.003187
9,price + volume + GDELT,Model C - price + volume + GDELT | GDELT perce...,11,xgb_150_depth4_lr0p03_sub0p8_col0p7_child20_ga...,150,4,0.03,0.8,0.7,20,10.0,0.0,1.0,balanced,0.358984,0.366280,0.339972,0.349188,0.370607,0.398473,0.368631,0.362245,Model B - price + volume | volume percentile r...,0.395999,0.002475
